In [6]:
pip install streamlit pandas matplotlib seaborn openpyxl

Note: you may need to restart the kernel to use updated packages.


In [11]:
import streamlit as st
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

@st.cache_data
def load_data():
    salary_df = pd.read_excel("Tab3_zpl_2024.xlsx")
    salary_old_df = pd.read_csv("Tab3_zpl_2000.csv", sep=';')
    inflation_df = pd.read_excel("inflation.xlsx")

    inflation_df = inflation_df[['Год', 'Всего']]
    inflation_df.columns = ['year', 'inflation']
    inflation_df = inflation_df.dropna()
    inflation_df['year'] = inflation_df['year'].astype(int)
    inflation_df['inflation'] = inflation_df['inflation'].astype(float)
    inflation_df = inflation_df.sort_values('year')
    inflation_df['cumulative_index'] = (1 + inflation_df['inflation'] / 100).cumprod()
    inflation_df['deflator'] = inflation_df['cumulative_index'].iloc[-1] / inflation_df['cumulative_index']

    salary_df = salary_df.dropna(subset=['Вид деятельности'])
    selected_sectors = [
        'государственное управление и обеспечение военной безопасности; социальное обеспечение',
        'деятельность в области информации и связи',
        'деятельность финансовая и страховая'
    ]
    mask = salary_df['Вид деятельности'].str.lower().str.contains('|'.join([s.lower() for s in selected_sectors]))
    salary_df = salary_df[mask]
    salary_long = salary_df.melt(id_vars='Вид деятельности', var_name='year', value_name='salary_nominal')
    salary_long['year'] = salary_long['year'].astype(int)
    salary_long = salary_long.reset_index(drop=True)

    salary_old_df = salary_old_df.dropna(subset=['Вид деятельности'])
    selected_old_sectors = [
        'Государственное управление и обеспечение военной безопасности; социальное страхование',
        'Финансовая деятельность',
        'из них связь'
    ]
    mask_old = salary_old_df['Вид деятельности'].str.lower().str.contains('|'.join([s.lower() for s in selected_old_sectors]))
    salary_old_df = salary_old_df[mask_old]
    salary_old_long = salary_old_df.melt(id_vars='Вид деятельности', var_name='year', value_name='salary_nominal')
    salary_old_long['year'] = salary_old_long['year'].astype(int)
    salary_old_long = salary_old_long.reset_index(drop=True)

    def mapping(x):
        if x == "Государственное управление и обеспечение военной безопасности; социальное страхование":
            return "государственное управление и обеспечение военной безопасности; социальное обеспечение"
        elif x == "Финансовая деятельность":
            return "деятельность финансовая и страховая"
        elif x == "из них связь":
            return "деятельность в области информации и связи"
        return x

    salary_old_long['Вид деятельности'] = salary_old_long['Вид деятельности'].apply(mapping)

    combined_salary_df = pd.concat([salary_old_long, salary_long], ignore_index=True)
    combined_salary_df = combined_salary_df.drop_duplicates(subset=['Вид деятельности', 'year'])
    combined_salary_df = combined_salary_df.sort_values(by=['Вид деятельности', 'year']).reset_index(drop=True)

    merged_df = combined_salary_df.merge(inflation_df[['year', 'deflator']], on='year', how='left')
    merged_df['salary_real'] = merged_df['salary_nominal'] * merged_df['deflator']

    return merged_df

data = load_data()

st.title("Анализ зарплат по отраслям (2000–2024 гг.)")

sectors = data['Вид деятельности'].unique()
selected_sector = st.selectbox("Выберите отрасль", sectors)

salary_type = st.radio("Выберите тип зарплаты:", ('Номинальная', 'Реальная'))

plot_data = data[data['Вид деятельности'] == selected_sector]

fig, ax = plt.subplots(figsize=(10, 6))
if salary_type == 'Номинальная':
    sns.lineplot(data=plot_data, x='year', y='salary_nominal', marker='o', ax=ax)
    ax.set_ylabel("Номинальная зарплата (₽)")
else:
    sns.lineplot(data=plot_data, x='year', y='salary_real', marker='o', ax=ax)
    ax.set_ylabel("Реальная зарплата (₽, цены 2024 года)")

ax.set_title(f"{salary_type} зарплата: {selected_sector}")
ax.set_xlabel("Год")
ax.grid(True)

st.pyplot(fig)

st.markdown("### Ключевые показатели")

col_name = 'salary_nominal' if salary_type == 'Номинальная' else 'salary_real'

st.write(f"Минимальная {salary_type.lower()} зарплата: {plot_data[col_name].min():.2f} ₽")
st.write(f"Максимальная {salary_type.lower()} зарплата: {plot_data[col_name].max():.2f} ₽")
st.write(f"Средняя {salary_type.lower()} зарплата: {plot_data[col_name].mean():.2f} ₽")

2025-06-12 20:53:23.060 No runtime found, using MemoryCacheStorageManager
2025-06-12 20:53:23.064 No runtime found, using MemoryCacheStorageManager
2025-06-12 20:53:23.065 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-12 20:53:23.066 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-12 20:53:23.067 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-12 20:53:23.256 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-12 20:53:23.257 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-12 20:53:23.257 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-06-12 20:53:23.258 Thread 'MainThread': missing ScriptRunContext! This warning can be ignor